# aether — live session (checkpoint from Drive)
Connects a live `aether serve` backend running in this Colab GPU runtime to a local microphone/speaker client over a Cloudflare quick tunnel, since a remote Colab VM has no public IP of its own.

Requires a **remote managed paid Colab runtime**. A BF16 GPU with at least 22 GiB free VRAM is required for inference (L4 or A100 class); this notebook does not train, so the 38 GiB adaptation floor from `aether_colab.ipynb` does not apply here. `CHECKPOINT_RUN_ID` and `CHECKPOINT_STEP` below select an existing adapter checkpoint already saved on Drive by `aether_colab.ipynb`; leaving `CHECKPOINT_STEP` empty serves the unadapted base model instead.

The Cloudflare quick tunnel launched here is unauthenticated and disposable: anyone with the printed URL can connect for as long as the tunnel stays open. Setting `LIVE_TOKEN` below requires a matching bearer token from the connecting client.

In [ ]:
import json
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

CONFIRM_REMOTE_PAID_COLAB = False
REPO_URL = "https://github.com/karl4th/aether-v2.git"
GIT_REF = "main"  # A commit/branch/tag; ideally the exact revision that produced the checkpoint.
UV_VERSION = "0.12.13"
DRIVE_ROOT = "/content/drive/MyDrive/aether"
RUN_ID = "live-" + datetime.now(UTC).strftime("%Y%m%d-%H%M%S")
CHECKPOINT_RUN_ID = "english-demo-20260918-141108"  # Existing run holding the checkpoint.
CHECKPOINT_STEP = "step-000020"  # Empty string "" serves the unadapted base model instead.
PORT = 8080
LIVE_TOKEN = ""  # Optional bearer token; the client passes it via AETHER_LIVE_TOKEN.
BUDGET_UNITS = 500

## Environment and exact checkout
Confirmation of the remote paid environment selection is required, matching `aether_colab.ipynb`. For a private GitHub repository, a read-only `GITHUB_TOKEN` Secret is used; the token is never stored in the URL or in the Git configuration.

In [ ]:
if not CONFIRM_REMOTE_PAID_COLAB:
    raise RuntimeError("Confirm managed remote paid Colab; local runtime is forbidden")
__import__("google.colab")
if sys.platform != "linux" or not Path("/content").is_dir():
    raise RuntimeError("Select a remote Colab Linux GPU runtime")

In [ ]:
import base64
import os
import tempfile

from google.colab import userdata


def checkout_source(repo_url, git_ref, workspace, git_env=None):
    if not git_ref or git_ref.startswith("-"):
        raise ValueError("Expected a Git branch, tag or commit SHA")
    project = Path(tempfile.mkdtemp(prefix="aether-src-", dir=workspace))
    env = dict(os.environ if git_env is None else git_env)
    env["GIT_TERMINAL_PROMPT"] = "0"

    def git(*args):
        result = subprocess.run(
            ["git", *args],
            cwd=project,
            env=env,
            check=True,
            capture_output=True,
            text=True,
            timeout=180,
        )
        return result.stdout.strip()

    git("init", "--quiet")
    git("remote", "add", "origin", repo_url)
    git("fetch", "--depth=1", "origin", git_ref)
    revision = git("rev-parse", "FETCH_HEAD^{commit}")
    git("checkout", "--detach", revision)
    for name in ("pyproject.toml", "uv.lock", ".python-version"):
        if not (project / name).is_file():
            raise ValueError("Missing required project file: " + name)
    return project, revision


# Optional read-only GitHub credential; never printed or stored in Git config.
git_env = os.environ.copy()
try:
    token = userdata.get("GITHUB_TOKEN")
except userdata.SecretNotFoundError:
    token = None
if token:
    if REPO_URL != "https://github.com/karl4th/aether-v2.git":
        raise ValueError("Credential use is restricted to the aether repository")
    auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
    git_env.update(
        {
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
            "GIT_CONFIG_VALUE_0": "Authorization: Basic " + auth,
        }
    )
    del auth
try:
    PROJECT, SOURCE_REVISION = checkout_source(REPO_URL, GIT_REF, "/content", git_env)
finally:
    git_env.clear()
    del token
print("Source commit:", SOURCE_REVISION)
print("Project:", PROJECT)

## Installation only in the remote environment
uv uses the lock file, Python 3.12.14, and a separate `.venv`. The `model` group brings in the pinned backend (PyTorch, moshi, etc.); the `audio` group is not needed here, since the microphone/speaker client runs on the local machine, not inside Colab.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv==" + UV_VERSION], check=True)
UV = [sys.executable, "-m", "uv"]
subprocess.run(
    UV + ["sync", "--locked", "--no-dev", "--group", "model", "--python", "3.12.14"],
    cwd=PROJECT,
    check=True,
)


def package_run(*arguments):
    result = subprocess.run(
        UV + ["run", "--locked", "--no-dev", "--group", "model", *map(str, arguments)],
        cwd=PROJECT,
        text=True,
        capture_output=True,
    )
    if result.returncode:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError("aether operation failed; inspect the diagnostic above")
    return result


print(package_run("aether", "--version").stdout)

## Runtime permit and storage
The permit is bound to the boot ID and the live notebook process, valid for up to 12 hours; it guards against accidental local execution, not proof of the billing tier. `allow_training` is always `False` here: this notebook only serves inference, it never constructs an optimizer. Logs and the run manifest are written to a new run directory on Google Drive.

In [ ]:
import hashlib
import os
import time

from google.colab import drive

drive.mount("/content/drive")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive is not mounted")
PERMIT = PROJECT / "runtime-permit.json"
now = time.time()
PERMIT.write_text(
    json.dumps(
        {
            "schema_version": 1,
            "boot_id": Path("/proc/sys/kernel/random/boot_id").read_text().strip(),
            "kernel_pid": os.getpid(),
            "created_at": now,
            "expires_at": now + 12 * 3600,
            "user_confirmed_remote_paid": CONFIRM_REMOTE_PAID_COLAB,
            "allow_training": False,
            "budget_units": BUDGET_UNITS,
            "source_revision": SOURCE_REVISION,
        }
    )
)
result = package_run(
    "python",
    "-c",
    "import sys; from pathlib import Path; from aether.storage import create_run; "
    "print(create_run(Path(sys.argv[1]),sys.argv[2]))",
    DRIVE_ROOT,
    RUN_ID,
)
RUN_DIR = Path(result.stdout.strip())
report = json.loads(
    package_run(
        "python",
        "-c",
        "import json; from aether.preflight import collect_preflight; "
        "print(json.dumps(collect_preflight()))",
    ).stdout
)
report.update(
    source_revision=SOURCE_REVISION,
    source_repository=REPO_URL,
    uv_lock_sha256=hashlib.sha256((PROJECT / "uv.lock").read_bytes()).hexdigest(),
    budget_units=BUDGET_UNITS,
    training_requested=False,
)
(RUN_DIR / "run.json").write_text(json.dumps(report, indent=2))
print(report["gpu"])
package_run(
    "python",
    "-c",
    "import sys; from aether.remote import require_remote_runtime; "
    "require_remote_runtime(permit_path=sys.argv[1]); import torch; "
    "from aether.backend import check_resources; "
    "check_resources(torch, training=False)",
    PERMIT,
)
print("Resource admission passed for inference; actual peak memory is checked once serving starts.")
backend_check = (
    "from aether.backend_contract import validate_backend_api; print(validate_backend_api())"
)
print(package_run("python", "-c", backend_check).stdout)

## Checkpoint selection
`CHECKPOINT_RUN_ID`/`CHECKPOINT_STEP` point at an existing, `COMPLETE`-marked checkpoint saved on Drive by `aether_colab.ipynb`. Its provenance (code revision, base model, dataset identity) is checked before it is ever handed to the live backend; a mismatched or corrupted checkpoint is rejected here rather than after the tunnel is already open.

In [ ]:
CHECKPOINT_PATH = None
if CHECKPOINT_STEP:
    CHECKPOINT_PATH = (
        Path(DRIVE_ROOT) / "runs" / CHECKPOINT_RUN_ID / "checkpoints" / CHECKPOINT_STEP
    )
    package_run(
        "python",
        "-c",
        "import sys; from pathlib import Path; from aether.storage import verify_checkpoint; "
        "verify_checkpoint(Path(sys.argv[1]))",
        CHECKPOINT_PATH,
    )
    print("Checkpoint verified:", CHECKPOINT_PATH)
else:
    print("No checkpoint selected; serving the unadapted base model.")

## Cloudflare quick tunnel
Installs `cloudflared` once per runtime. The quick tunnel needs no account, no DNS record, and no inbound firewall change; it is disposable and unauthenticated by itself, which is why `LIVE_TOKEN` above matters.

In [ ]:
subprocess.run(
    [
        "wget",
        "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O",
        "/usr/local/bin/cloudflared",
    ],
    check=True,
)
subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
version = subprocess.run(
    ["/usr/local/bin/cloudflared", "--version"], capture_output=True, text=True
)
print(version.stdout)

## Start the live backend
`aether serve` loads the pinned weights (and the checkpoint above, if any), then listens on `127.0.0.1` only — nothing here is reachable without the tunnel. It runs in the background for the rest of this session; its own stdout/stderr are kept in `serve.log` under this run's Drive directory.

In [ ]:
import urllib.error
import urllib.request

if LIVE_TOKEN:
    os.environ["AETHER_LIVE_TOKEN"] = LIVE_TOKEN
serve_args = UV + [
    "run",
    "--locked",
    "--no-dev",
    "--group",
    "model",
    "aether",
    "serve",
    "--permit",
    str(PERMIT),
    "--port",
    str(PORT),
]
if CHECKPOINT_PATH is not None:
    serve_args += ["--checkpoint", str(CHECKPOINT_PATH)]
SERVER_LOG = RUN_DIR / "serve.log"
server_log_handle = SERVER_LOG.open("w")
SERVER_PROCESS = subprocess.Popen(
    serve_args, cwd=PROJECT, stdout=server_log_handle, stderr=subprocess.STDOUT
)

deadline = time.time() + 300  # First load can take a while; later runs use the local cache.
ready = False
while time.time() < deadline:
    if SERVER_PROCESS.poll() is not None:
        server_log_handle.close()
        print(SERVER_LOG.read_text()[-4000:])
        raise RuntimeError("aether serve exited before becoming ready; see the log above")
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health/live", timeout=2) as response:
            if response.status == 200:
                ready = True
                break
    except (urllib.error.URLError, ConnectionError):
        pass
    time.sleep(2)
if not ready:
    raise TimeoutError("aether serve did not become live within 300s; check serve.log")
print("Live backend is up. PID:", SERVER_PROCESS.pid, "Log:", SERVER_LOG)

## Expose it and connect
This cell blocks and prints the tunnel's public URL; it keeps running so the tunnel stays open for the session. Stop this cell (Colab's interrupt/stop button) to close the tunnel — the live backend keeps running in the background and this cell can be re-run to expose it again.

From a local machine with a microphone and speakers (not from inside Colab):
```bash
uv run --locked --group audio aether talk --url wss://<printed-host>/v1/session
```
Test in headphones first: echo cancellation has not been implemented, so speaker playback can feed back into the microphone.

In [ ]:
tunnel_args = UV + [
    "run",
    "--locked",
    "--no-dev",
    "--group",
    "model",
    "aether",
    "tunnel",
    "--port",
    str(PORT),
]
try:
    subprocess.run(tunnel_args, cwd=PROJECT)
finally:
    print("Tunnel cell stopped. The live backend is still running (PID", SERVER_PROCESS.pid, ").")
    print("Re-run this cell to reopen a tunnel, or run the next cell to stop the backend.")

## Stop the live backend
Run explicitly when finished, then disconnect the GPU runtime to stop idle resource consumption.

In [ ]:
if SERVER_PROCESS.poll() is None:
    SERVER_PROCESS.terminate()
    try:
        SERVER_PROCESS.wait(timeout=10)
    except subprocess.TimeoutExpired:
        SERVER_PROCESS.kill()
        SERVER_PROCESS.wait(timeout=10)
server_log_handle.close()
print("Live backend stopped. Results and the log are in:", RUN_DIR)
print("Disconnect the GPU runtime when finished to stop idle resource consumption.")